# Philadelphia CIP – Image-Based PDF Parser (2021–2025)

Parses image-based Philadelphia CIP PDFs (2021–2025) into per-year CSVs.
Skips 2019 and 2020 (text-based, handled separately).

**Dependencies:** `pip install pymupdf pdfplumber pytesseract pillow opencv-python-headless pandas`

Tesseract must be installed and on PATH.
Windows: also set `pytesseract.pytesseract.tesseract_cmd`.

## Cell 1 – Imports & paths

In [27]:
import re
import warnings
from pathlib import Path

import cv2
import numpy as np
import pdfplumber
import fitz          # pymupdf
import pytesseract
import pandas as pd
from PIL import Image

warnings.filterwarnings('ignore')

# Windows: set this if tesseract is not on PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

SCRIPTS_DIR = Path('.').resolve()
CITY_DIR    = SCRIPTS_DIR.parent / 'Philadelphia'
PDF_DIR     = CITY_DIR / 'PDF'
CSV_DIR     = CITY_DIR / 'CSV'
CSV_DIR.mkdir(exist_ok=True)

SKIP_STEMS = {'2019', '2020'}

REQUIRED_COLS = [
    'cip_year', 'project_type', 'source_page', 'department',
    'project_name', 'project_id', 'address_location',
    'start_year', 'end_year',
    'project_description', 'project_justification',
    'previous_appropriations', 'project_total',
]

SRC_CODES = {
    'CN', 'CT', 'CR', 'CA', 'A',
    'XN', 'XT', 'XR', 'Z',
    'FB', 'FO', 'FT',
    'PB', 'PT',
    'SB', 'SO', 'ST',
    'TB', 'TO', 'TT',
    'GO', 'GR', 'GN', 'GP',
}

YEAR_RE  = re.compile(r'^(\d{4})$')
NUM_RE   = re.compile(r'^-?[\d,]+$')
COMBINED = re.compile(r'^[\d,]+[A-Z]{1,3}$')
PROJ_RE  = re.compile(r'^(\d+[A-Z]?)\.?\s+(.+)$')
DEPT_RE  = re.compile(r'^[A-Z][A-Z &\-/()\u2013]+')

TESS_CFG = '--psm 6 --oem 1'
OCR_DPI  = 300

pdfs_to_process = sorted(
    p for p in PDF_DIR.glob('*.pdf') if p.stem not in SKIP_STEMS
)
print('Paths OK.')
print(f'  PDF_DIR : {PDF_DIR}')
print(f'  CSV_DIR : {CSV_DIR}')
print('PDFs to process:', [p.name for p in pdfs_to_process])

Paths OK.
  PDF_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\PDF
  CSV_DIR : C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\CSV
PDFs to process: ['2021.pdf', '2022.pdf', '2023.pdf', '2024.pdf', '2025.pdf']


## Cell 2 – Page classification

In [28]:
def classify_pages(pdf_path, min_page=15, max_text_chars=300):
    candidates = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, pg in enumerate(pdf.pages):
            pg_num = i + 1
            if pg_num < min_page:
                continue
            if len(pg.images) >= 1 and len(pg.chars) < max_text_chars:
                candidates.append(pg_num)
    return candidates


def is_data_page(ocr_text):
    years_found = len(re.findall(r'\b20\d{2}\b', ocr_text))
    has_unit    = bool(re.search(r'x000|\$x|x 000', ocr_text, re.I))
    return years_found >= 4 and has_unit


for p in pdfs_to_process:
    print(f'{p.name}: {len(classify_pages(p))} candidate image pages')

2021.pdf: 182 candidate image pages
2022.pdf: 178 candidate image pages
2023.pdf: 190 candidate image pages
2024.pdf: 193 candidate image pages
2025.pdf: 212 candidate image pages


## Cell 3 – Image extraction & OCR

In [29]:
def _ocr_image(pdf_path, pg_1idx, dpi=OCR_DPI):
    doc   = fitz.open(str(pdf_path))
    mat   = fitz.Matrix(dpi / 72, dpi / 72)
    pix   = doc[pg_1idx - 1].get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
    doc.close()
    arr   = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    arr   = clahe.apply(arr)
    _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(arr)


def _layout_lines(data):
    grouped = {}
    n = len(data.get('text', []))
    for i in range(n):
        text = str(data['text'][i]).strip()
        if not text:
            continue
        try:
            conf = float(data.get('conf', ['-1'])[i])
        except Exception:
            conf = -1
        if conf < 0:
            continue
        word = {
            'text': text,
            'left': int(data['left'][i]),
            'top': int(data['top'][i]),
            'width': int(data['width'][i]),
            'height': int(data['height'][i]),
            'conf': conf,
        }
        word['cx'] = word['left'] + word['width'] / 2
        word['cy'] = word['top'] + word['height'] / 2
        key = (data.get('block_num', [0])[i], data.get('par_num', [0])[i], data.get('line_num', [0])[i])
        grouped.setdefault(key, []).append(word)

    lines = []
    for words in grouped.values():
        words = sorted(words, key=lambda w: w['left'])
        lines.append({
            'words': words,
            'text': ' '.join(w['text'] for w in words),
            'left': min(w['left'] for w in words),
            'right': max(w['left'] + w['width'] for w in words),
            'top': min(w['top'] for w in words),
            'bottom': max(w['top'] + w['height'] for w in words),
        })
    return sorted(lines, key=lambda ln: (ln['top'], ln['left']))


def ocr_page_layout(pdf_path, pg_1idx, dpi=OCR_DPI):
    img = _ocr_image(pdf_path, pg_1idx, dpi=dpi)
    data = pytesseract.image_to_data(img, config=TESS_CFG, output_type=pytesseract.Output.DICT)
    lines = _layout_lines(data)
    text = '\n'.join(line['text'] for line in lines)
    return {'text': text, 'lines': lines, 'width': img.width, 'height': img.height}


def ocr_page(pdf_path, pg_1idx, dpi=OCR_DPI):
    return ocr_page_layout(pdf_path, pg_1idx, dpi=dpi)['text']


# Uncomment to spot-check OCR quality:
# for p in pdfs_to_process:
#     cands = classify_pages(p)
#     if cands:
#         text = ocr_page(p, cands[0])
#         print(f'\n{p.name}  page {cands[0]}:')
#         for line in text.split('\n')[:10]:
#             if line.strip(): print(' ', repr(line.strip()))

print('OCR helpers ready.')

OCR helpers ready.


## Cell 4 – Line classifier

In [30]:
DEPT_KEYWORDS = {
    'MUSEUM', 'AVIATION', 'COMMERCE', 'FINANCE', 'FIRE', 'FLEET',
    'LIBRARY', 'HEALTH', 'HOMELESS', 'HOUSING', 'SERVICES',
    'LICENSES', 'MANAGING', 'MURAL', 'PARKS', 'PLANNING', 'POLICE',
    'PRISONS', 'PROPERTY', 'RECORDS', 'REVENUE', 'SCHOOLS',
    'SEPTA', 'STREETS', 'TRANSIT', 'WATER', 'ZOO', 'OIT',
    'SUSTAINABILITY', 'MDO', 'OHS',
}

def is_src(tok):
    return tok.upper() in SRC_CODES

def parse_num(s):
    try:
        return float(str(s).replace(',', '').strip())
    except Exception:
        return None

def is_department(text):
    return any(kw in text.upper() for kw in DEPT_KEYWORDS)

def classify_line(line, plan_years):
    line = line.strip()
    if not line or re.match(r'^\d{1,3}$', line):
        return ('skip',)
    if re.search(r'x000|\$x|x 000', line, re.I):
        return ('unit',)
    tokens = line.split()
    yrs = [int(t) for t in tokens if YEAR_RE.match(t) and 2018 <= int(t) <= 2040]
    if len(yrs) >= 4:
        return ('year_hdr', list(dict.fromkeys(yrs)))
    if re.match(r'^TOTALS?\s*[-\u2013]', line, re.I):
        return ('totals_hdr',)
    m = PROJ_RE.match(line)
    if m:
        name = m.group(2).strip()
        first_name_tok = name.split()[0] if name.split() else ''
        if (
            re.search(r'[A-Za-z]', name)
            and not re.search(r'\btotal\b', name, re.I)
            and not is_src(first_name_tok)
            and not NUM_RE.match(first_name_tok.replace(',', ''))
            and not COMBINED.match(first_name_tok)
            and not YEAR_RE.match(first_name_tok)
        ):
            return ('project', m.group(1).rstrip('.'), name)
    nb_start = None
    for i, tok in enumerate(tokens):
        if NUM_RE.match(tok.replace(',', '')) or is_src(tok) or COMBINED.match(tok):
            nb_start = i
            break
    if nb_start is not None:
        left_text = ' '.join(tokens[:nb_start]).strip()
        right     = tokens[nb_start:]
        has_sc    = any(is_src(t) or COMBINED.match(t) for t in right)
        nums = []
        for t in right:
            if COMBINED.match(t):
                nums.append(parse_num(re.sub(r'[A-Z]+$', '', t)))
            elif NUM_RE.match(t.replace(',', '')):
                nums.append(parse_num(t))
        nums = [n for n in nums if n is not None]
        if not nums:
            return ('text', line)
        if has_sc:
            return ('src_row', nums, left_text)
        yr_vals = {plan_years[j]: v for j, v in enumerate(nums[:-1]) if j < len(plan_years)}
        range_total = nums[-1] if len(nums) > 1 else (nums[0] if nums else None)
        return ('total_row', yr_vals, range_total, left_text)
    m = PROJ_RE.match(line)
    if m:
        name = m.group(2).strip()
        if not re.search(r'\btotal\b', name, re.I):
            return ('project', m.group(1).rstrip('.'), name)
    if DEPT_RE.match(line) and len(line.replace(' ', '')) > 3 and 'TOTAL' not in line.upper():
        return ('dept', line)
    return ('text', line)

def _could_be_split_project_title(line):
    line = line.strip()
    if not line:
        return False
    if re.match(r'^\d', line):
        return False
    if re.search(r'x000|\$x|x 000', line, re.I):
        return False
    if re.match(r'^TOTALS?\s*[-\u2013]', line, re.I):
        return False
    tokens = line.split()
    years = [t for t in tokens if YEAR_RE.match(t) and 2018 <= int(t) <= 2040]
    if len(years) >= 2:
        return False
    if tokens and (is_src(tokens[0]) or NUM_RE.match(tokens[0].replace(',', '')) or COMBINED.match(tokens[0])):
        return False
    if DEPT_RE.match(line) and line.upper() == line:
        return False
    return True

def logical_ocr_lines(ocr_text):
    lines = [ln.strip() for ln in ocr_text.split('\n') if ln.strip()]
    i = 0
    while i < len(lines):
        line = lines[i]
        if re.match(r'^\d{1,3}\.?$', line) and i + 1 < len(lines):
            nxt = lines[i + 1]
            if _could_be_split_project_title(nxt):
                yield f'{line.rstrip(".")} {nxt}'
                i += 2
                continue
        yield line
        i += 1

def logical_layout_lines(lines):
    i = 0
    while i < len(lines):
        line = lines[i]
        text = line.get('text', '').strip()
        if re.match(r'^\d{1,3}\.?$', text) and i + 1 < len(lines):
            nxt = lines[i + 1]
            nxt_text = nxt.get('text', '').strip()
            if _could_be_split_project_title(nxt_text):
                merged = dict(nxt)
                merged['words'] = line.get('words', []) + nxt.get('words', [])
                merged['text'] = f'{text.rstrip(".")} {nxt_text}'
                merged['left'] = min(line.get('left', 0), nxt.get('left', 0))
                merged['right'] = max(line.get('right', 0), nxt.get('right', 0))
                merged['top'] = min(line.get('top', 0), nxt.get('top', 0))
                merged['bottom'] = max(line.get('bottom', 0), nxt.get('bottom', 0))
                yield merged
                i += 2
                continue
        yield line
        i += 1

def _median(values):
    values = sorted(v for v in values if v is not None)
    if not values:
        return None
    mid = len(values) // 2
    return values[mid] if len(values) % 2 else (values[mid - 1] + values[mid]) / 2

def detect_column_centers(layout, plan_years):
    centers = {}
    if not layout or not layout.get('lines'):
        return centers
    page_h = layout.get('height') or 1
    header_lines = []
    for line in layout.get('lines', []):
        if line.get('top', 0) > page_h * 0.35:
            continue
        found = set()
        for word in line.get('words', []):
            digits = re.sub(r'\D', '', word['text'])
            if digits.isdigit() and int(digits) in plan_years:
                found.add(int(digits))
        if found:
            header_lines.append((len(found), line))
    header_line = max(header_lines, key=lambda item: item[0])[1] if header_lines else None
    for year in plan_years:
        xs = []
        if header_line:
            xs = [word['cx'] for word in header_line.get('words', []) if re.sub(r'\D', '', word['text']) == str(year)]
        if not xs:
            for line in layout.get('lines', []):
                if line.get('top', 0) > page_h * 0.35:
                    continue
                for word in line.get('words', []):
                    if re.sub(r'\D', '', word['text']) == str(year):
                        xs.append(word['cx'])
        x = min(xs) if xs else None
        if x is not None:
            centers[year] = x
    year_xs = [centers[y] for y in plan_years if y in centers]
    gaps = [b - a for a, b in zip(year_xs, year_xs[1:]) if b > a]
    gap = _median(gaps)
    if year_xs and gap:
        centers['total'] = year_xs[-1] + gap
    return centers

def _amount_source_cells(line):
    words = line.get('words', []) if isinstance(line, dict) else []
    cells = []
    i = 0
    while i < len(words):
        raw = words[i]['text'].replace('$', '')
        raw = raw.strip('.,;:')
        combo = re.match(r'^(-?[\d,]+)([A-Za-z]{1,3})$', raw)
        if combo and is_src(combo.group(2)):
            value = parse_num(combo.group(1))
            if value is not None:
                cells.append({'value': value, 'src': combo.group(2).upper(), 'x': words[i]['cx'], 'left': words[i]['left']})
            i += 1
            continue
        raw_num = raw.replace('O', '0') if not re.search(r'[A-Za-z]', raw) else raw
        if NUM_RE.match(raw_num) and i + 1 < len(words):
            src = words[i + 1]['text'].strip('.,;:').upper()
            if is_src(src):
                value = parse_num(raw_num)
                if value is not None:
                    cells.append({'value': value, 'src': src, 'x': words[i]['cx'], 'left': words[i]['left']})
                i += 2
                continue
        i += 1
    return cells

def _numeric_cells(line):
    cells = []
    words = line.get('words', []) if isinstance(line, dict) else []
    for word in words:
        raw = word['text'].replace('$', '').strip('.,;:')
        raw = raw.replace('O', '0') if not re.search(r'[A-Za-z]', raw) else raw
        if NUM_RE.match(raw):
            value = parse_num(raw)
            if value is not None:
                cells.append({'value': value, 'x': word['cx'], 'left': word['left']})
    return cells

def _assign_cells_to_columns(cells, centers, plan_years):
    if not cells or not centers:
        return {}, None
    keys = [y for y in plan_years if y in centers]
    if 'total' in centers:
        keys.append('total')
    if not keys:
        return {}, None
    xs = [centers[k] for k in keys]
    gaps = [b - a for a, b in zip(sorted(xs), sorted(xs)[1:]) if b > a]
    max_dist = max(25, (_median(gaps) or 90) * 0.55)
    year_vals = {}
    total = None
    for cell in cells:
        nearest = min(keys, key=lambda key: abs(cell['x'] - centers[key]))
        if abs(cell['x'] - centers[nearest]) > max_dist:
            continue
        if nearest == 'total':
            total = (total or 0) + cell['value']
        else:
            year_vals[nearest] = year_vals.get(nearest, 0) + cell['value']
    return year_vals, total

def classify_layout_line(line, plan_years, ctx):
    text = line.get('text', '') if isinstance(line, dict) else str(line)
    base = classify_line(text, plan_years)
    if base[0] in ('year_hdr', 'unit', 'totals_hdr', 'dept', 'project', 'skip'):
        return base
    centers = ctx.get('column_centers') or {}
    if isinstance(line, dict) and centers:
        src_cells = _amount_source_cells(line)
        if src_cells:
            year_vals, row_total = _assign_cells_to_columns(src_cells, centers, plan_years)
            left_edge = min(c['left'] for c in src_cells)
            left_text = ' '.join(w['text'] for w in line.get('words', []) if w['left'] < left_edge).strip()
            if year_vals or row_total is not None:
                return ('src_row_spatial', year_vals, row_total, left_text)
        nums = _numeric_cells(line)
        if len(nums) >= 2:
            year_vals, row_total = _assign_cells_to_columns(nums, centers, plan_years)
            left_edge = min(c['left'] for c in nums)
            left_text = ' '.join(w['text'] for w in line.get('words', []) if w['left'] < left_edge).strip()
            if year_vals or row_total is not None:
                return ('total_row', year_vals, row_total, left_text)
    return base

print('Line classifier ready.')

Line classifier ready.


## Cell 5 – Page parser

**Project hierarchy tracking:**

Philadelphia CIPs nest projects: a parent project line (e.g. `74 SEPTA Bridge...`) acts as a title header with no funding rows of its own. Sub-projects (`1`, `2`, `3` ...) follow immediately underneath and carry the actual source/total rows.

The resulting `project_id` format is `DEPT.parent.child` (e.g. `DEPT.74.1`).

Detection: if `pending` has no `pending_yr` and no `has_funding_rows` when the next project line arrives, the pending entry is a header. Its number becomes `main_proj_num`, and sub-project IDs are built as `DEPT.main_proj_num.child_num`.

In [31]:
def _flush(ctx, projects):
    pending    = ctx.get('pending')
    pending_yr = ctx.get('pending_yr') or ctx.get('pending_src_yr')
    if pending is None:
        return
    if not pending_yr:
        return  # no year values; project may continue on next page
    name = pending.get('project_name', '').strip()
    if name and len(name) >= 3 and name[0] not in ("'", '|', '"', '.', ','):
        for yr, v in pending_yr.items():
            pending[f'year_{yr}'] = v
        rt = ctx.get('pending_tot') if ctx.get('pending_tot') is not None else ctx.get('pending_src_tot')
        pending['project_total']         = rt if rt else sum(pending_yr.values())
        pending['project_description']   = ' '.join(ctx.get('desc_buf', [])).strip()
        pending['project_justification'] = pending['project_description']
        projects.append(pending)
    ctx['pending']          = None
    ctx['pending_yr']       = None
    ctx['pending_tot']      = None
    ctx['pending_src_yr']   = None
    ctx['pending_src_tot']  = None
    ctx['pending_src_bad']  = False
    ctx['desc_buf']         = []
    ctx['has_funding_rows'] = False


def _hard_flush(ctx, projects):
    if ctx.get('pending_yr') or ctx.get('pending_src_yr'):
        _flush(ctx, projects)
    else:
        ctx['pending']          = None
        ctx['pending_yr']       = None
        ctx['pending_tot']      = None
        ctx['pending_src_yr']   = None
        ctx['pending_src_tot']  = None
        ctx['pending_src_bad']  = False
        ctx['desc_buf']         = []
        ctx['has_funding_rows'] = False


def _is_header(ctx):
    # A pending project with no funding rows is a parent title line.
    return (
        ctx.get('pending') is not None
        and not ctx.get('pending_yr')
        and not ctx.get('pending_src_yr')
        and not ctx.get('has_funding_rows')
    )


def _add_source_values(ctx, nums, plan_years):
    if not nums:
        return
    values = nums[:-1] if len(nums) > 1 else nums
    row_total = nums[-1] if len(nums) > 1 else nums[0]
    if len(nums) > 1 and row_total:
        value_sum = sum(values)
        if abs(value_sum - row_total) > max(2, abs(row_total) * 0.05):
            ctx['pending_src_bad'] = True
            return
    src_yr = ctx.get('pending_src_yr') or {}
    for idx, value in enumerate(values):
        if idx >= len(plan_years):
            break
        year = plan_years[idx]
        src_yr[year] = src_yr.get(year, 0) + value
    ctx['pending_src_yr'] = src_yr
    ctx['pending_src_tot'] = (ctx.get('pending_src_tot') or 0) + row_total


def _add_source_year_values(ctx, year_vals, row_total):
    if not year_vals and row_total is None:
        return
    if row_total is not None and year_vals:
        value_sum = sum(year_vals.values())
        if abs(value_sum - row_total) > max(2, abs(row_total) * 0.05):
            ctx['pending_src_bad'] = True
            return
    src_yr = ctx.get('pending_src_yr') or {}
    for year, value in year_vals.items():
        src_yr[year] = src_yr.get(year, 0) + value
    ctx['pending_src_yr'] = src_yr
    if row_total is not None:
        ctx['pending_src_tot'] = (ctx.get('pending_src_tot') or 0) + row_total


def parse_page(ocr_page_result, pg_num, cip_year, ctx, projects):
    plan_years = ctx.get('plan_years', [])
    if isinstance(ocr_page_result, dict):
        ctx['column_centers'] = detect_column_centers(ocr_page_result, plan_years) or ctx.get('column_centers', {})
        page_lines = logical_layout_lines(ocr_page_result.get('lines', []))
    else:
        page_lines = logical_ocr_lines(ocr_page_result)
    for raw_line in page_lines:
        cat = classify_layout_line(raw_line, plan_years if plan_years else list(range(2022, 2028)), ctx)

        if cat[0] == 'year_hdr':
            new_years = cat[1][:6]
            if new_years != plan_years:
                plan_years = new_years
                ctx['plan_years'] = plan_years

        elif cat[0] in ('unit', 'skip'):
            pass

        elif cat[0] == 'totals_hdr':
            _hard_flush(ctx, projects)
            ctx['main_proj_num'] = ''  # reset hierarchy at section totals

        elif cat[0] == 'dept':
            _hard_flush(ctx, projects)
            txt = cat[1].strip().upper()
            if is_department(txt):
                ctx['dept']          = txt
                ctx['proj_type']     = ''
                ctx['main_proj_num'] = ''  # reset hierarchy on new department
            else:
                ctx['proj_type'] = cat[1].strip()

        elif cat[0] == 'src_row':
            _, nums, left_text = cat
            if ctx.get('pending') is not None:
                ctx['has_funding_rows'] = True
                _add_source_values(ctx, nums, plan_years)
                if left_text:
                    ctx['desc_buf'].append(left_text)

        elif cat[0] == 'src_row_spatial':
            _, yr_vals, row_total, left_text = cat
            if ctx.get('pending') is not None:
                ctx['has_funding_rows'] = True
                _add_source_year_values(ctx, yr_vals, row_total)
                if left_text:
                    ctx['desc_buf'].append(left_text)

        elif cat[0] == 'total_row':
            _, yr_vals, range_total, left_text = cat
            pending = ctx.get('pending')
            if pending is not None and (ctx.get('pending_src_yr') or ctx.get('pending_src_bad')):
                continue
            if pending is not None and yr_vals and ctx.get('pending_yr') is None:
                if not (range_total and sum(yr_vals.values()) > range_total * 3):
                    ctx['has_funding_rows'] = True
                    ctx['pending_yr']  = yr_vals
                    ctx['pending_tot'] = range_total
                    _flush(ctx, projects)

        elif cat[0] == 'project':
            _, num_str, name = cat
            dept    = ctx.get('dept', '')
            dept_id = re.sub(r'\s+', '', dept)[:5] if dept else 'UNK'

            # If pending has seen no funding rows, it is a parent header.
            # Record its number as main_proj_num before flushing.
            if _is_header(ctx):
                pid_str    = ctx['pending'].get('project_id', '')
                parent_num = pid_str.rsplit('.', 1)[-1] if '.' in pid_str else pid_str
                ctx['main_proj_num'] = parent_num

            _flush(ctx, projects)

            main = ctx.get('main_proj_num', '')

            # Numeric comparison: incoming > parent -> new top-level project
            # incoming <= parent -> sub-project under that parent
            try:
                num_int  = int(re.sub(r'[A-Za-z]', '', num_str))
                main_int = int(re.sub(r'[A-Za-z]', '', main)) if main else 0
                has_suffix = bool(re.search(r'[A-Za-z]', num_str))
                is_top   = (not main) or (num_int > main_int) or (has_suffix and num_int == main_int)
            except (ValueError, TypeError):
                is_top = not main

            if is_top:
                pid = f'{dept_id}.{num_str}'
                ctx['main_proj_num'] = ''  # will be set only if confirmed header
            else:
                pid = f'{dept_id}.{main}.{num_str}'

            ctx['pending'] = {
                'cip_year': cip_year, 'project_type': ctx.get('proj_type', ''),
                'source_page': pg_num, 'department': dept,
                'project_name': name, 'project_id': pid,
                'address_location': '', 'start_year': '', 'end_year': '',
                'project_description': '', 'project_justification': '',
                'previous_appropriations': '', 'project_total': '',
            }
            ctx['has_funding_rows'] = False
            ctx['pending_yr']  = None
            ctx['pending_tot'] = None
            ctx['pending_src_yr']  = None
            ctx['pending_src_tot'] = None
            ctx['pending_src_bad'] = False
            ctx['desc_buf']    = []

        elif cat[0] == 'text':
            if cat[1].strip() and ctx.get('pending') is not None:
                ctx['desc_buf'].append(cat[1].strip())


print('Page parser ready.')

Page parser ready.


## Cell 6 – PDF-level parser

In [32]:
def parse_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    stem     = pdf_path.stem
    cip_year = int(stem) if stem.isdigit() else None
    plan_years = []
    with pdfplumber.open(pdf_path) as pdf:
        for pg in pdf.pages[:30]:
            txt = pg.extract_text() or ''
            m   = re.search(
                r'(?:FY|FISCAL YEARS?)\s*(\d{4})\s*[-\u2013]\s*(\d{4})',
                txt, re.I,
            )
            if m:
                y1, y2 = int(m.group(1)), int(m.group(2))
                plan_years = list(range(y1, min(y2 + 1, y1 + 7)))[:6]
                if cip_year is None: cip_year = y1 - 1
                break
    if not plan_years and cip_year:
        plan_years = list(range(cip_year + 1, cip_year + 7))
    print(f'  cip_year={cip_year}  plan_years={plan_years}')
    image_pages = classify_pages(pdf_path)
    total = len(image_pages)
    print(f'  candidate image pages: {total}')
    ctx = {
        'plan_years':       plan_years,
        'dept':             '',
        'proj_type':        '',
        'main_proj_num':    '',   # current parent project number for hierarchy
        'has_funding_rows': False,
        'pending':          None,
        'pending_yr':       None,
        'pending_tot':      None,
        'pending_src_yr':   None,
        'pending_src_tot':  None,
        'pending_src_bad':  False,
        'desc_buf':         [],
    }
    projects = []; data_pages = 0
    for i, pg_num in enumerate(image_pages, 1):
        if i % 20 == 0:
            print(f'  OCR: {i}/{total} (page {pg_num})...', flush=True)
        try:
            ocr_result = ocr_page_layout(pdf_path, pg_num)
        except Exception as exc:
            print(f'  OCR error page {pg_num}: {exc}'); continue
        if not is_data_page(ocr_result['text']): continue
        data_pages += 1
        parse_page(ocr_result, pg_num, cip_year, ctx, projects)
    _flush(ctx, projects)
    print(f'  data pages processed: {data_pages}')
    return cip_year, plan_years, projects


print('PDF parser ready.')

PDF parser ready.


## Cell 7 – DataFrame builder

In [33]:
def build_dataframe(projects, plan_years):
    if not projects:
        return pd.DataFrame(columns=REQUIRED_COLS)
    year_cols = [f'year_{y}' for y in sorted(plan_years)]
    all_cols  = REQUIRED_COLS + year_cols
    df = pd.DataFrame(projects)
    for col in all_cols:
        if col not in df.columns:
            df[col] = ''
    # Derive start_year / end_year; cast to str to match column dtype
    for idx, row in df.iterrows():
        active = {
            y: row.get(f'year_{y}')
            for y in plan_years
            if pd.notna(row.get(f'year_{y}')) and row.get(f'year_{y}', 0) not in ('', 0, 0.0)
        }
        if active:
            df.at[idx, 'start_year'] = str(min(active))
            df.at[idx, 'end_year']   = str(max(active))
    return df[all_cols].fillna('')

print('DataFrame builder ready.')

DataFrame builder ready.


## Cell 8 – Main loop: parse all PDFs → write CSVs

In [25]:
summary = []

for pdf_path in pdfs_to_process:
    print(f'\n== {pdf_path.name} ==')
    cip_year, plan_years, projects = parse_pdf(pdf_path)
    print(f'  rows extracted: {len(projects)}')
    df       = build_dataframe(projects, plan_years)
    csv_path = CSV_DIR / f'{pdf_path.stem}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'  -> {csv_path}')
    for col in REQUIRED_COLS:
        if col in df.columns:
            null_pct = (df[col].astype(str).str.strip() == '').mean() * 100
            if null_pct > 50:
                print(f'    WARN {col}: {null_pct:.0f}% empty')
    summary.append({
        'pdf': pdf_path.name, 'cip_year': cip_year,
        'plan_years': plan_years, 'rows': len(df), 'csv': str(csv_path),
    })

print('\n== ALL DONE ==')
for s in summary:
    print(f"{s['pdf']}  ->  {s['rows']} rows  plan_years={s['plan_years']}")


== 2021.pdf ==
  cip_year=2021  plan_years=[2022, 2023, 2024, 2025, 2026, 2027]
  candidate image pages: 182
  OCR: 20/182 (page 66)...
  OCR: 40/182 (page 87)...
  OCR: 60/182 (page 116)...
  OCR: 80/182 (page 147)...
  OCR: 100/182 (page 167)...
  OCR: 120/182 (page 187)...
  OCR: 140/182 (page 213)...
  OCR: 160/182 (page 239)...
  OCR: 180/182 (page 265)...
  data pages processed: 164
  rows extracted: 601
  -> C:\Users\vince\Documents\GitHub\CIPBD\Philadelphia\CSV\2021.csv
    WARN address_location: 100% empty
    WARN previous_appropriations: 100% empty

== 2022.pdf ==
  cip_year=2022  plan_years=[2023, 2024, 2025, 2026, 2027, 2028]
  candidate image pages: 178
  OCR: 20/178 (page 57)...
  OCR: 40/178 (page 77)...
  OCR: 60/178 (page 103)...
  OCR: 80/178 (page 134)...
  OCR: 100/178 (page 154)...
  OCR: 120/178 (page 175)...
  OCR: 140/178 (page 203)...
  OCR: 160/178 (page 225)...
  data pages processed: 169
  rows extracted: 585
  -> C:\Users\vince\Documents\GitHub\CIPBD\Phil

## Cell 9 – Output validation

In [26]:
print('=== Validation Summary ===\n')
for s in summary:
    csv_path = Path(s['csv'])
    if not csv_path.exists():
        print(f'{csv_path.name}: FILE MISSING'); continue
    df = pd.read_csv(csv_path, dtype=str)
    year_cols = [c for c in df.columns if c.startswith('year_')]
    print(f'{csv_path.name}')
    print(f'  Rows         : {len(df)}')
    print(f'  Year columns : {year_cols}')
    print(f'  Depts unique : {df["department"].nunique()}')
    print(f'  Missing name : {(df["project_name"].str.strip() == "").sum()}')
    print(f'  Missing total: {(df["project_total"].astype(str).str.strip() == "").sum()}')
    sample_cols = ['department', 'project_id', 'project_name', 'project_total'] + year_cols[:3]
    with pd.option_context('display.max_colwidth', 40):
        print(df[sample_cols].head(3).to_string(index=False))
    print()

=== Validation Summary ===

2021.csv
  Rows         : 601
  Year columns : ['year_2022', 'year_2023', 'year_2024', 'year_2025', 'year_2026', 'year_2027']
  Depts unique : 20
  Missing name : 0
  Missing total: 0
                   department project_id                                         project_name project_total year_2022 year_2023 year_2024
ART MUSEUM COMPLICN - CAPITAL    ARTMU.1 Philadelphia Museum of Art - Building Rehabilitatian        2500.0    2900.0    1000.0    1000.0
                     AVIATION  AVIAT.2.1                                        Airfield Area      201826.0   71100.0   23193.0   26125.0
                     AVIATION  AVIAT.3.1                                        Terminal Area      194258.0   83750.0   20137.0   21273.0

2022.csv
  Rows         : 585
  Year columns : ['year_2023', 'year_2024', 'year_2025', 'year_2026', 'year_2027', 'year_2028']
  Depts unique : 21
  Missing name : 0
  Missing total: 0
                  department project_id            